# Partition uncertainties in projections of impacts

1. Of just mean estimates
2. With regression uncertainties
3. Bootstrapped over number of runs (maybe spin that off into a 4b?)
4. TODO: Without MERRA2

For different sets of runs: 
- a "balanced panel" of GCMs / runs
- all available runs at GWL 2n

In [23]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from matplotlib import colors as mcolors

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle

dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38573 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/38573/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/38573/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42785,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/38573/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43879,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/42721/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:34249,


2026-02-18 14:55:21,299 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 27fd3aad4e0c169ea2e0040996ce91da initialized by task ('rechunk-merge-rechunk-transfer-ab9bc4633826ef14b2234fc499a051e0', 0, 0, 0, 0, 0, 0, 1, 0) executed on worker tcp://127.0.0.1:37559
2026-02-18 14:57:07,906 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 27fd3aad4e0c169ea2e0040996ce91da deactivated due to stimulus 'task-finished-1771451827.9045124'
2026-02-18 14:57:11,502 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 27fd3aad4e0c169ea2e0040996ce91da initialized by task ('rechunk-merge-rechunk-transfer-ab9bc4633826ef14b2234fc499a051e0', 0, 0, 0, 0, 0, 0, 1, 0) executed on worker tcp://127.0.0.1:33027
2026-02-18 14:57:12,944 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 27fd3aad4e0c169ea2e0040996ce91da deactivated due to stimulus 'task-finished-1771451832.9424312'
2026-02-18 16:18:07,185 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7d95132cee8b46c0

## Climate uncertainty partitioning

In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

In [48]:
overwrite = False
for proc in params_proc:
    output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}.zarr'
    if overwrite or (not os.path.exists(output_fn)):
        #---------------------------
        # Setup
        #---------------------------
        # Load impacts projections
        fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}*.zarr'))
        def load_impact(fn):
            ds_tmp = xr.open_zarr(fn,consolidated=False)
            ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
            ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})
        
            ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
            ds_tmp = ds_tmp.reset_index('idv')
            ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
            return ds_tmp
        
        with warnings.catch_warnings():
            # Silence zarr warnings related to the /.done flags used
            warnings.filterwarnings('ignore')
            dss = xr.concat([load_impact(fn) for fn in fns],
                        dim='idv',join='outer')
        
        
        dss = dss.chunk({'poly_idx':500,'idv':-1,'proj_base':-1,'gwl':-1})
        
        # Figure out how many BCD-ME members reach a given GWL
        gwl_counter = dss.maxgwl.load().reset_index('idv').groupby(['model','maxgwl']).count()
        gwl_counter = gwl_counter.sel(maxgwl = gwl_counter.maxgwl.values[::-1]).cumsum('maxgwl')
        #print(gwl_counter)
        
        uncerts_all = []
        #---------------------------
        # Uncertainty partitioning over all runs
        #---------------------------
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
        
        uncerts = uncerts.expand_dims({'sample':['allruns']})
        
        uncerts_all.append(uncerts)
        
        #---------------------------
        # Uncertainty partitioning over a balanced panel up to GWL 3
        #---------------------------
        max_gwl = 3
        nruns = 13
        
        gwlmods = gwl_counter.sel(maxgwl=3)
        proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
        
        dss_tmp = dss.isel(idv = dss.maxgwl>=max_gwl).groupby('model').map(lambda x: x.isel(idv=slice(0, nruns)))
        dss_tmp = dss_tmp.isel(idv=[m in proc_mods for m in dss_tmp.model.values]).chunk({'idv':-1})
        dss_tmp = dss_tmp.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
        
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
        
        uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods']})
        
        uncerts_all.append(uncerts)

        #---------------------------
        # Uncertainty partitioning over a balanced panel up to GWL 3
        # without MERRA2
        #---------------------------
        max_gwl = 3
        nruns = 13
        
        gwlmods = gwl_counter.sel(maxgwl=3)
        proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
        
        dss_tmp = dss.isel(idv = dss.maxgwl>=max_gwl).groupby('model').map(lambda x: x.isel(idv=slice(0, nruns)))
        dss_tmp = dss_tmp.sel(proj_base = [pb for pb in dss_tmp.proj_base.values if pb != 'MERRA2'])
        dss_tmp = dss_tmp.isel(idv=[m in proc_mods for m in dss_tmp.model.values]).chunk({'idv':-1})
        dss_tmp = dss_tmp.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
        
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
        
        uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods_noMERRA2']})
        
        uncerts_all.append(uncerts)
        
        #---------------------------
        # Export
        #---------------------------
        # Concatenate into single ds
        uncerts_all = xr.concat(uncerts_all,dim='sample')
        uncerts_all = uncerts_all.where(uncerts_all.sum('typ')>0)
        
        # Export
        uncerts_all.attrs = {'SOURCE':'4_partition_uncertainty.ipynb',
                             'DESCRIPTION':f'Uncertainty partitioning of projections of {proc} between model, internal, and reanalysis sources'}
        
        utility_save(uncerts_all,output_fn,zarr_mode = 'w')
    else:
        print(output_fn+' exists!')
    


/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_mortality.zarr exists!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_gdppc.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_payroll.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Climate and regression uncertainty partitioning